## 依赖安装

```powershell
python -m pip install tensorflow torch transformers
```

TensorFlow 和 PyTorch 示例不需要下载预训练模型。Transformers 示例使用本地 BERT。

In [5]:
import os
import numpy as np
np.random.seed(42)
print("NumPy版本：",np.__version__)

NumPy版本： 2.0.2


## 1. TensorFlow / Keras 模型

In [6]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    tensorflow_ready=True; print("TensorFlow版本：",tf.__version__)
except ImportError:
    tensorflow_ready=False; print("TensorFlow未安装。")

TensorFlow版本： 2.20.0


In [7]:
if tensorflow_ready:
    keras_model=keras.Sequential([layers.Input(shape=(784,)),layers.Dense(64,activation="relu"),layers.Dense(10,activation="softmax")])
    keras_model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"]); keras_model.summary()
else: print("跳过Keras模型。")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,890 (198.79 KB)

 Trainable params: 50,890 (198.79 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
if tensorflow_ready:
    x_train=np.random.randn(64,784).astype("float32"); y_train=np.random.randint(0,10,size=(64,))
    history=keras_model.fit(x_train,y_train,epochs=2,batch_size=16,verbose=1)
    print("最终Loss：",history.history["loss"][-1]); print("输出形状：",keras_model(x_train[:3]).shape)
else: print("TensorFlow未安装。")

Epoch 1/2
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.0833 - loss: 2.8420  
Epoch 2/2
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4854 - loss: 1.5936 
最终Loss： 1.593231439590454
输出形状： (3, 10)


## 2. PyTorch 模型

页面示例在 `forward()` 末尾先做 Softmax，然后再使用 `CrossEntropyLoss`。实际训练时，`CrossEntropyLoss` 应直接接收未归一化 Logits，因此下面移除模型内部 Softmax。

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
class Net(nn.Module):
    def __init__(self): super().__init__(); self.fc1=nn.Linear(784,64); self.fc2=nn.Linear(64,10)
    def forward(self,x): return self.fc2(torch.relu(self.fc1(x)))

torch_model=Net(); criterion=nn.CrossEntropyLoss(); optimizer=optim.Adam(torch_model.parameters())
print(torch_model)

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Net(
  (fc1): Linear(in_features=784, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=10, bias=True)
)


In [10]:
x=torch.randn(32,784); y=torch.randint(0,10,(32,))
logits=torch_model(x); loss=criterion(logits,y)
optimizer.zero_grad(); loss.backward(); optimizer.step()
print("Logits形状：",logits.shape); print("损失：",loss.item()); print("第一层梯度范数：",torch_model.fc1.weight.grad.norm().item())

Logits形状： torch.Size([32, 10])
损失： 2.352297067642212
第一层梯度范数： 1.9120233058929443


## 3. PyTorch 自动微分

In [11]:
a=torch.tensor(2.0,requires_grad=True); b=a**3+2*a; b.backward()
print("b=",b.item()); print("db/da=",a.grad.item()); print("理论结果3a²+2=",3*a.item()**2+2)

b= 12.0
db/da= 14.0
理论结果3a²+2= 14.0


In [12]:
from transformers import AutoTokenizer,AutoModel

transformer_path=r"D:\11\NLP\data\bert-base-uncased-local"
transformer_ready=os.path.isdir(transformer_path) and os.path.isfile(os.path.join(transformer_path,"config.json")) and any(os.path.isfile(os.path.join(transformer_path,name)) for name in ["model.safetensors","pytorch_model.bin"])
print("本地Transformer模型准备完成：",transformer_ready)

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


本地Transformer模型准备完成： True


In [13]:
if transformer_ready:
    transformer_tokenizer=AutoTokenizer.from_pretrained(transformer_path,local_files_only=True)
    transformer_model=AutoModel.from_pretrained(transformer_path,local_files_only=True); transformer_model.eval()
    transformer_inputs=transformer_tokenizer("Hello world!",return_tensors="pt")
    with torch.no_grad(): transformer_outputs=transformer_model(**transformer_inputs)
    print("Tokens：",transformer_tokenizer.convert_ids_to_tokens(transformer_inputs["input_ids"][0])); print("输出形状：",transformer_outputs.last_hidden_state.shape)
else: print("请先准备本地bert-base-uncased模型。")

Tokens： ['[CLS]', 'hello', 'world', '!', '[SEP]']
输出形状： torch.Size([1, 5, 768])


## 5. PyTorch 动态量化：本地模型优化示例

In [14]:
quantized_model=torch.ao.quantization.quantize_dynamic(torch_model,{nn.Linear},dtype=torch.qint8)
with torch.no_grad(): original_output=torch_model(x[:2]); quantized_output=quantized_model(x[:2])
print("原模型输出形状：",original_output.shape); print("量化模型输出形状：",quantized_output.shape); print("平均绝对差：",(original_output-quantized_output).abs().mean().item())

原模型输出形状： torch.Size([2, 10])
量化模型输出形状： torch.Size([2, 10])
平均绝对差： 0.0025452501140534878


C:\Users\Administrator\AppData\Local\Temp\ipykernel_5324\3148490626.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model=torch.ao.quantization.quantize_dynamic(torch_model,{nn.Linear},dtype=torch.qint8)


## 6. TensorFlow Lite 转换：使用刚才本地创建的 Keras 模型

In [15]:
if tensorflow_ready:
    saved_model_dir=r"D:\11\NLP\data\keras_saved_model_local"; os.makedirs(saved_model_dir,exist_ok=True)
    tf.saved_model.save(keras_model,saved_model_dir); converter=tf.lite.TFLiteConverter.from_saved_model(saved_model_dir); converter.optimizations=[tf.lite.Optimize.DEFAULT]; tflite_quant_model=converter.convert()
    tflite_path=r"D:\11\NLP\data\keras_model_quant.tflite"; open(tflite_path,"wb").write(tflite_quant_model); print("TFLite文件：",tflite_path); print("大小：",os.path.getsize(tflite_path)/1024,"KB")
else: print("TensorFlow未安装，跳过TFLite。")

INFO:tensorflow:Assets written to: D:\11\NLP\data\keras_saved_model_local\assets


INFO:tensorflow:Assets written to: D:\11\NLP\data\keras_saved_model_local\assets


TFLite文件： D:\11\NLP\data\keras_model_quant.tflite
大小： 2.83203125 KB
